# Pandas, Polars, GeoPandas a GeoPolars

Notebook střídá shodné kroky v pandas a polars, následně ukazuje práci s geopandas a geopolars.

In [ ]:
import geopandas as gpd
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import polars as pl
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from plotnine import aes, geom_point, ggplot, labs, scale_color_brewer, theme_bw
from utils import data_path, save_data_path

diamonds_url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"
countries_file = data_path("ne_10m_admin_0_countries.shp")

## Pandas: načtení tabulkových dat

In [ ]:
# Načtení tabulkových dat pro pandas i polars
df_pd = pd.read_csv(diamonds_url)

print(f"Pandas shape: {df_pd.shape}")
print(df_pd.head(5))
print(df_pd.dtypes)

## Polars: načtení tabulkových dat

In [ ]:
df_pl = pl.read_csv(diamonds_url)

print(f"Polars shape: {df_pl.shape}")
print(df_pl.head(5))
print(df_pl.dtypes)

## Pandas: výpočty, filtrace a agregace

In [ ]:
# Výpočet odvozených sloupců
df_pd_work = df_pd.copy()
df_pd_work["volume"] = df_pd_work["x"] * df_pd_work["y"] * df_pd_work["z"]
df_pd_work["price_per_volume"] = df_pd_work["price"] / df_pd_work["volume"]
df_pd_work["price_per_volume"] = df_pd_work["price_per_volume"].replace(
    [float("inf"), -float("inf")], float("nan")
)

# Filtrace a agregace
filtered_pd = df_pd_work[df_pd_work["price"] > 2000]
grouped_pd = (
    filtered_pd.groupby(["cut", "color"])["price"]
    .mean()
    .reset_index(name="mean_price")
    .sort_values(by=["color", "cut"])
    .reset_index(drop=True)
)

print(df_pd_work[["x", "y", "z", "volume", "price_per_volume"]].head())
print(grouped_pd.head(10))

## Polars: výpočty, filtrace a agregace

In [ ]:
df_pl_work = (
    df_pl.with_columns((pl.col("x") * pl.col("y") * pl.col("z")).alias("volume"))
    .with_columns((pl.col("price") / pl.col("volume")).alias("price_per_volume"))
    .with_columns(
        pl.when(pl.col("price_per_volume").is_infinite())
        .then(None)
        .otherwise(pl.col("price_per_volume"))
        .alias("price_per_volume")
    )
)

filtered_pl = df_pl_work.filter(pl.col("price") > 2000)
grouped_pl = (
    filtered_pl.group_by(["cut", "color"])
    .agg(pl.col("price").mean().alias("mean_price"))
    .sort(["color", "cut"])
)

print(df_pl_work.select(["x", "y", "z", "volume", "price_per_volume"]).head())
print(grouped_pl.head(10))

In [ ]:
# Kontrola shody výsledků pandas vs polars
grouped_pl_pd = grouped_pl.to_pandas()
comparison = grouped_pd.merge(grouped_pl_pd, on=["cut", "color"], suffixes=("_pandas", "_polars"))
comparison["abs_diff"] = (comparison["mean_price_pandas"] - comparison["mean_price_polars"]).abs()

print(f"Rows compared: {len(comparison)}")
print(f"Max absolute difference: {comparison['abs_diff'].max()}")
print(comparison.head(10))

## Vizualizace

In [ ]:
# Vykreslení pandas dat
plot_pd = (
    ggplot(df_pd_work, aes(x="carat", y="price", color="color"))
    + geom_point(alpha=0.5, size=1)
    + labs(title="Dependence of Price on Carat (pandas)", x="Carat", y="Price", color="Color")
    + scale_color_brewer(type="qual", palette="Dark2")
    + theme_bw()
)

plot_pd

In [ ]:

# Vykreslení polars dat (po převodu do pandas)
plot_pl = (
    ggplot(df_pl_work.to_pandas(), aes(x="carat", y="price", color="color"))
    + geom_point(alpha=0.5, size=1)
    + labs(title="Dependence of Price on Carat (polars)", x="Carat", y="Price", color="Color")
    + scale_color_brewer(type="qual", palette="Dark2")
    + theme_bw()
)

plot_pl

In [ ]:
# Uložení obou grafů
plot_pd_path = save_data_path("price_vs_carat_pandas_notebook.png", delete_if_exist=True)
plot_pl_path = save_data_path("price_vs_carat_polars_notebook.png", delete_if_exist=True)
plot_pd.save(str(plot_pd_path), dpi=300)
plot_pl.save(str(plot_pl_path), dpi=300)
print(f"Saved: {plot_pd_path.as_posix()}")
print(f"Saved: {plot_pl_path.as_posix()}")

## GeoPandas: prostorová data

In [ ]:
gdf = gpd.read_file(countries_file)

print(f"GeoPandas řádky: {len(gdf)}")
print(f"GeoPandas CRS: {gdf.crs}")
print(gdf[["NAME", "CONTINENT", "POP_EST"]].head(10))

europe_gdf = gdf[gdf["CONTINENT"] == "Europe"].copy()
europe_projected = europe_gdf.to_crs(epsg=3035)
europe_projected["area_km2"] = europe_projected.geometry.area / 1e6

print(europe_projected[["NAME", "area_km2"]].sort_values("area_km2", ascending=False).head(10))

In [ ]:
# Jednoduchá mapa Evropy
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
europe_projected.plot(
    column="POP_EST",
    cmap="YlOrRd",
    legend=True,
    legend_kwds={"label": "Population", "shrink": 0.6, "format": "%.0f"},
    edgecolor="black",
    linewidth=0.5,
    ax=ax,
)

# Dodatečné formátování legendy bez vědeckého zápisu
colorbar_axis = fig.axes[-1]
colorbar_axis.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))

# Jednoduché měřítko: délka 1000 km (EPSG:3035 má jednotky v metrech)
scale_bar = AnchoredSizeBar(
    ax.transData,
    1_000_000,
    "1000 km",
    loc="lower left",
    pad=0.4,
    color="black",
    frameon=True,
    size_vertical=15_000,
    fontproperties=fm.FontProperties(size=10),
)
ax.add_artist(scale_bar)
ax.set_title("European Countries by Population", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Export výsledku do GeoPackage
result_file = save_data_path("europe_countries_notebook.gpkg", delete_if_exist=True)
europe_projected.to_file(result_file, driver="GPKG", layer="countries")
print(f"Saved: {result_file.as_posix()}")